In [1]:
import io
import re
import pandas as pd
import numpy as np
import requests

from playwright.async_api import async_playwright, Locator, Page, expect
from pathlib import Path
from pypdf import PdfReader

In [2]:
title_csv = "../dataset/sipora_title.csv"
content_csv = "../dataset/sipora_content_human.csv"

output_dir = Path("../dataset")
output_dir.mkdir(exist_ok=True)

In [3]:
# Get titles

years_count = 5
start_year = 2016
items_per_year_count = 200

async def get_titles():
    p = await async_playwright().start()
    browser = await p.chromium.launch()
    page = await browser.new_page()
    
    with open(title_csv, "w") as f:
        f.write('"title","link","year"')
    
        for i in range(years_count):
            year = start_year + i
            await page.goto(f"https://sipora.polije.ac.id/view/year/{year}.html")
            await expect(page).to_have_title(re.compile("SIPORA"))
    
            content = page.locator(".ep_view_page")
            items = await content.locator("p").all()
            anchors: list[Locator] = []
            for item in items:
                text_content = await item.text_content()
                if text_content is not None and "thesis" in text_content:
                    anchors.append(item.locator("a"))
    
            links_count = 0
            for anchor in anchors:
                title = await anchor.text_content()
                link = await anchor.get_attribute("href")
    
                if link is None or title is None:
                    continue
    
                if links_count >= items_per_year_count:
                    break
    
                title = title.replace("\n", " ").replace('"', "")
                f.write(f'\n"{title}","{link}","{year}"')
                links_count += 1
                
    print("All title of thesis saved!")

await get_titles()

All title of thesis saved!


In [4]:
async def get_contents():
    df = pd.read_csv(title_csv)
    links = df['link']
    success_count = 0
    total = len(links)
    
    p = await async_playwright().start()
    browser = await p.chromium.launch()
    page = await browser.new_page()

    with open(content_csv, "w") as f:
        f.write('"link","content","type","lang","label"')
    
        for link in links:
            if link is not None and type(link) is str:
                await page.goto(link)
                await expect(page).to_have_title(re.compile("SIPORA"))
    
                pdf_link_locators = await page.locator(
                    ".ep_summary_content_main>table>tbody>tr>td>a.ep_document_link"
                ).all()
                already_fetched_links: list[str] = []
    
                for pdf_link_locator in pdf_link_locators:
                    pdf_link = await pdf_link_locator.get_attribute("href")
                    if not pdf_link:
                        continue
    
                    if pdf_link in already_fetched_links:
                        continue
    
                    if "lengkap" in pdf_link.lower():
                        continue
    
                    if (
                        "pustaka" in pdf_link.lower()
                        or "bibliography" in pdf_link.lower()
                        or "referensi" in pdf_link.lower()
                        or "reference" in pdf_link.lower()
                    ):
                        continue
    
                    try:
                        response = requests.get(pdf_link)
                        already_fetched_links.append(pdf_link)
                        file_stream = io.BytesIO(response.content)
                        reader = PdfReader(file_stream)
    
                        current_buffer: str = ""
    
                        for p in reader.pages:
                            text = p.extract_text()
                            rtype = "content"
                            if (
                                "abstract" in pdf_link.lower()
                                or "abstrak" in pdf_link.lower()
                            ):
                                rtype = "abstract"
                            if (
                                "summary" in pdf_link.lower()
                                or "ringkasan" in pdf_link.lower()
                            ):
                                rtype = "summary"
                            lang = (
                                "en"
                                if (
                                    "the" in text.strip().lower()
                                    or " a " in text.strip().lower()
                                )
                                else "id"
                            )
    
                            for text in text.split("\n"):
                                if (
                                    text.strip() == ""
                                    or len(text.strip().split(" ")) < 3
                                    or re.match(r"[A-Z]+\s[A-Z]+", text.strip())
                                    or re.match(r"BAB [0-9]*\.?\s.*", text.strip())
                                    or re.match(
                                        r"[0-9]*(\.[0-9]\.?){1,}\s.*", text.strip()
                                    )
                                ):
                                    continue
    
                                current_buffer += text
                                if (
                                    re.match(r".*[.?!:]$", current_buffer.strip())
                                    and len(current_buffer.strip().split(".")) > 1
                                    and not current_buffer.strip()
                                    .lower()
                                    .endswith(" rp.")
                                ):
                                    f.write(
                                        f'\n"{link}","{current_buffer.strip().replace("\"", "\"\"")}","{rtype}","{lang}","human"'
                                    )
                                    current_buffer = ""
    
                            current_buffer = ""
    
                    except Exception as _:
                        continue
    
                    success_count += 1
    print(f"Successfully extracted {success_count}/{total} contents!")

await get_contents()

invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found


Successfully extracted 1869/941 contents!
